# msCBCT: SIRT reconstruction

SIRT reconstruction of a **measured multi-source CBCT scan** on the exact ASTRA `cone_vec` geometry of the scanner. It is the same geometry `../msCBCT` trains NAF on, so the two volumes overlay voxel for voxel.

* **Acquisition**: 8 sources along z fire in turn, each onto its own detector band (65 rows at 0.4 mm), while the gantry keeps rotating. Frame `k` is fired by source `k % 8` at `START_DEG + (k // 8) * SPAN_DEG / (N / 8) + (k % 8) * deg_per_frame`, with `deg_per_frame` = 0.125 deg from `params.py`.
* **Geometry** comes from `params.py`: SOD / ODD, the half-fan offset `offset_u` and the z tables of the sources and their bands. The bin factor follows from the pixel width in the `.nii` header.
* **Input**: NIfTI with on-disk dims (v, u, frame) = (65, 372, 8 x views) at 0.4 mm, values are line integrals `-ln(I / I0)`, row index increasing with +z.
* **Run time**: about 4-5 min for 5760 frames x 300 iterations on an RTX 5090.

Outputs in `OUT_DIR`, with `<stem> = recon_<name>_SIRT_it<iterations>` and `<name>` the input file name without `.nii` (and without a leading `proj_`):

* `<stem>.nii`: linear attenuation coefficient mu [cm^-1]
* `<stem>_HU.nii`: Hounsfield units, `HU = (mu - mu_water) / (mu_water - mu_air) * 1000`, with mu_water / mu_air at `MEAN_ENERGY_KEV` (67 keV for the msCBCT beam)
* `<stem>_HU_dicom/`: the same HU as a CT DICOM series, one file per axial slice

The `.nii` files are float32 with NIfTI dims (x, y, z) and the voxel width in the header.

## 1. Imports

Run the notebook from this folder (`reconstruction/`), so that `astra_recon.py`, `hu.py` and `params.py` are importable.

In [ ]:
import astra
from astra_recon import ConeBeamReconstruction

print(f"ASTRA {astra.__version__}  |  CUDA: {astra.use_cuda()}")

## 2. Parameters (the only cell you normally edit)

* `SPAN_DEG` / `START_DEG` / `ROT_DIR` must match the scan (and the NAF training config's `rot_dir`).
* `IMG_SIZE = (240.0, 240.0, 120.0)` with `VOXEL_SIZE = 0.4` is the NAF training grid of 0.4 mm data, so the SIRT and NAF volumes can be compared voxel by voxel (`VOXEL_SIZE = None` uses the projection pixel width).
* `MEAN_ENERGY_KEV` is the mean energy of the beam, used only for mu -> HU: CBCT 95 keV, msCBCT 67 keV. `PHANTOM`, `KVP` and `TUBE_CURRENT_MA` only fill DICOM tags.

In [ ]:
PROJ_FILE  = "../input/Real_RANDO_msCBCT_04mm/proj_RANDO_msCBCT_360views.nii"
SPAN_DEG   = 360.0            # [deg] frame i sits at START_DEG + i * SPAN_DEG / N
START_DEG  = 0.0              # [deg]
ROT_DIR    = +1
IMG_SIZE   = (240.0, 240.0, 120.0)  # [mm] (x, y, z) — the NAF grid
VOXEL_SIZE = 0.4              # [mm]; None = projection pixel width
ITERATIONS = 300
OUT_DIR    = "./output"
GPU        = None
# ---- mu -> HU and DICOM tags ----
MEAN_ENERGY_KEV = 67.0        # [keV] mean beam energy of the scan: CBCT 95, msCBCT 67
PHANTOM         = None        # DICOM patient name / ID; None = the input file name
KVP             = 110         # DICOM tag
TUBE_CURRENT_MA = 11          # DICOM tag [mA]

## 3. Load the projection

Prints the geometry, frames, angles and volume size. **Check the angles line**: it must match the scan.

In [ ]:
rec = ConeBeamReconstruction.from_file(PROJ_FILE, "msCBCT", span_deg=SPAN_DEG,
                                       start_deg=START_DEG, rot_dir=ROT_DIR,
                                       img_size=IMG_SIZE, voxel_size=VOXEL_SIZE)

## 4. Reconstruct and show the central slices

SIRT3D_CUDA with a non-negativity constraint. The coronal and sagittal views show +z up.

In [ ]:
rec.run(iterations=ITERATIONS, gpu=GPU)
rec.show_slices()

## 5. Save

Writes mu (`.nii`), HU (`_HU.nii`) and the HU DICOM series (`_HU_dicom/`) to `OUT_DIR`.

In [ ]:
rec.save(OUT_DIR, MEAN_ENERGY_KEV, phantom=PHANTOM, kvp=KVP,
         tube_current_ma=TUBE_CURRENT_MA)